In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
brze_dim_carburant_df = spark.read.table("fuel_price_dev.landing.brze_dim_carburant")
brze_dim_geo_df       = spark.read.table("fuel_price_dev.landing.brze_dim_geo")
brze_dim_station_df   = spark.read.table("fuel_price_dev.landing.brze_dim_station")
brze_fait_prix_df     = spark.read.table("fuel_price_dev.landing.brze_fait_prix")
brze_fait_rupture_df  = spark.read.table("fuel_price_dev.landing.brze_fait_rupture")

In [0]:
s_dim_carburant_df = brze_dim_carburant_df\
            .withColumn("date_traitement", now())\
            .withColumn("id", col("id").cast(IntegerType()))\
            .select("id","nom","date_ingestion", "date_traitement")\
            .dropDuplicates(["id"])

s_dim_geo_df = brze_dim_geo_df\
            .withColumn("date_traitement_silver", now())\
            .select(
                "code_region",
                "region",
                "code_departement",
                "departement",
                "date_ingestion",
                "date_traitement"
        ).dropDuplicates(["id"])


s_dim_station_df = brze_dim_station_df\
                      .withColumn("date_traitement", now())\
                      .select(
                          "station_id",
                          "adresse",
                          "ville",
                          "code_postal",
                          "code_departement",
                          "latitude",
                          "longitude",
                          "service",
                          "code_region",
                          "date_ingestion",
                          "date_traitement"
                ) .dropDuplicates(["station_id"])

In [0]:
# ============================================================================
# BRONZE FACTS  TRANSFORMATIONS
# ============================================================================
s_fait_prix_df = brze_fait_prix_df\
                    .withColumn("date_traitement", now())\
                    .withColumn("carburant_id", col("carburant_id").cast(IntegerType()))\
                    .withColumn("date_maj", expr("try_cast(date_maj AS TIMESTAMP)"))\
                    .withColumn("prix", col("date_ingestion").cast(IntegerType()))\
                    .select("id_fct_pr","station_id", "carburant_id", "date_maj", "prix", "date_ingestion", "date_traitement")
                        
brze_fait_rupture_df = brze_fait_rupture_df\
                        .withColumn("date_traitement", now())\
                        .withColumn("id_carburant", col("id_carburant").cast(IntegerType()))\
                        .withColumn("debut_rupture", expr("try_cast(debut_rupture AS TIMESTAMP)"))\
                        .withColumn("fin_rupture", expr("try_cast(fin_rupture AS TIMESTAMP)"))\
                        .select("id_fct_rpt","station_id", "id_carburant", "debut_rupture", "fin_rupture", "type_rupture", "date_ingestion", "date_traitement")